# Fase 1 - Actividad 2: Limpieza Textual y Bifurcación Estructural

El objetivo de esta actividad es adecuar la información textual a los requerimientos específicos de cada arquitectura algorítmica. 
Se establece un flujo bifurcado:
- **Ruido Global**: Se remueven etiquetas HTML y URLs para todos los textos.
- **Flujo Conservador (BETO)**: Preserva capitalización, signos de puntuación y *stopwords*, esencial para la extracción de sintaxis y contexto del tokenizador WordPiece.
- **Flujo Estricto (Modelos Lineales)**: Aplica minúsculas y elimina *stopwords* y puntuación para minimizar la dimensionalidad de las matrices dispersas TF-IDF.

In [9]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import os

# Descarga de recursos léxicos
nltk.download('stopwords')
stop_words_es = set(stopwords.words('spanish'))

[nltk_data] Downloading package stopwords to C:\Users\Baller
[nltk_data]     293\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 1. Carga y Unificación del Texto Base

In [10]:
df_raw = pd.read_csv('data/PoliticalFakeNews_OG.csv', sep=';')

# Rellenar nulos de manera segura en caso de que un registro no tenga descripción
df_raw['Titulo'] = df_raw['Titulo'].fillna('')
df_raw['Descripcion'] = df_raw['Descripcion'].fillna('')

# Unificar en una sola cadena de texto consolidada
df_raw['texto_base'] = df_raw['Titulo'] + " " + df_raw['Descripcion']

print(f"Corpus consolidado: {df_raw.shape[0]} registros.")

Corpus consolidado: 57231 registros.


## 2. Definición de Funciones de Limpieza

In [11]:
def limpieza_global(texto):
    """Remueve ruido que no aporta valor a NINGÚN modelo (URLs, HTML)."""
    if not isinstance(texto, str):
        return ""
    # Eliminar URLs
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    # Eliminar etiquetas HTML
    texto = re.sub(r'<.*?>', '', texto)
    # Remover espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def limpieza_estricta(texto):
    """Flujo destructivo para reducir la dimensionalidad en TF-IDF (Modelos Lineales)."""
    if not isinstance(texto, str):
        return ""
    # 1. Minúsculas
    texto = texto.lower()
    # 2. Dejar solo caracteres alfabéticos (remueve puntuación y números)
    texto = re.sub(r'[^a-záéíóúñü]', ' ', texto)
    # 3. Eliminar stopwords y reconstruir el texto
    palabras = texto.split()
    palabras_limpias = [p for p in palabras if p not in stop_words_es]
    # 4. Unir y quitar espacios sobrantes
    texto = " ".join(palabras_limpias)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

## 3. Ejecución de la Bifurcación Estructural

In [12]:
print("1. Aplicando limpieza global...")
df_raw['texto_global'] = df_raw['texto_base'].apply(limpieza_global)

print("2. Generando flujo conservador (BETO)...")
# El flujo conservador es esencialmente el texto con limpieza global
df_raw['texto_beto'] = df_raw['texto_global']

print("3. Generando flujo estricto (Clasificadores Lineales)...")
# El flujo estricto aplica normalización extrema sobre el texto global
df_raw['texto_lineal'] = df_raw['texto_global'].apply(limpieza_estricta)

print("\n--- Ejemplo de Bifurcación ---")
print(f"ORIGINAL: {df_raw['texto_base'].iloc[0]}")
print(f"\nBETO (Conservador): {df_raw['texto_beto'].iloc[0]}")
print(f"LINEAL (Estricto):  {df_raw['texto_lineal'].iloc[0]}")

1. Aplicando limpieza global...
2. Generando flujo conservador (BETO)...
3. Generando flujo estricto (Clasificadores Lineales)...

--- Ejemplo de Bifurcación ---
ORIGINAL: Moreno intenta apaciguar el flanco sanitario mientras enreda con la fecha de las elecciones El presidente abre la puerta a unos comicios en junio que no sean en domingo.

BETO (Conservador): Moreno intenta apaciguar el flanco sanitario mientras enreda con la fecha de las elecciones El presidente abre la puerta a unos comicios en junio que no sean en domingo.
LINEAL (Estricto):  moreno intenta apaciguar flanco sanitario mientras enreda fecha elecciones presidente abre puerta comicios junio domingo


## 4. Reestructuración de Columnas y Exportación del Dataset Unificado

Corregimos la columna ID nativa que venía rota, descartamos columnas que no aportan al modelado (como la fecha original) y seleccionamos exclusivamente el ID numérico, la Etiqueta (Label) y ambas características textuales preprocesadas.

In [13]:
# Reemplazamos los IDs rotos ('ID', 'ID') por un correlativo numérico real (1, 2, 3...)
df_raw['id'] = range(1, len(df_raw) + 1)

# Descartamos la Fecha y los textos sueltos, nos quedamos solo con lo que van a consumir los modelos
columnas_finales = ['id', 'Label', 'texto_beto', 'texto_lineal']
df_clean = df_raw[columnas_finales].copy()
df_clean.rename(columns={'Label': 'label'}, inplace=True)

# Guardamos el CSV maestro
output_path = 'data/PoliticalFakeNews_Clean_Bifurcated.csv'
df_clean.to_csv(output_path, sep=';', index=False)

print(f"Dataset unificado guardado exitosamente en: {output_path}")
display(df_clean.head(5))

Dataset unificado guardado exitosamente en: data/PoliticalFakeNews_Clean_Bifurcated.csv


,id,label,texto_beto,texto_lineal
0,1,1,Moreno intenta apaciguar el flanco sanitario m...,moreno intenta apaciguar flanco sanitario mien...
1,2,1,La Abogacía del Estado se retira como acusació...,abogacía retira acusación pieza iberdrola caso...
2,3,0,Las promesas incumplidas de Pablo Echenique en...,promesas incumplidas pablo echenique sanidad e...
3,4,1,Sánchez defiende 'resolver el problema' de la ...,sánchez defiende resolver problema ley solo di...
4,5,1,Ian Gibson cierra la lista electoral de la con...,ian gibson cierra lista electoral confluencia ...
